# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mahmoonakhan/flyrank-ml-internship-task1/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Binary classification, operated as a ranked scoring task.**

The core ML problem is binary classification: predict whether a page is declining (`trend_direction == "down"`) or not. However, the operational output is not a simple yes/no — it is a **ranked score** (predicted probability of decline) that produces a review queue. A content reviewer looks at the top-scored pages first.

I chose this framing because:
- A classifier gives me a probability I can rank by, which maps directly to "review this page before that one."
- The starter pipeline proved this works: a random forest classifier achieved Precision@50 ≈ 0.74 vs. the baseline's 0.24.
- Ranking by probability is more useful than a hard threshold because reviewer capacity varies — some weeks they can review 20 pages, other weeks 50.

In [6]:
pass

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Current proxy label:** `is_declining_label = (trend_direction == "down")`

This is a **proxy**, not the ideal target. It is built from a 30-day vs 30-day impression comparison within the current window. It tells me what *already happened*, not what *will happen next*.

**Why it is good enough for now:**
- It is available in the starter dataset.
- It is cleanly defined and reproducible.
- It lets me build the end-to-end pipeline without waiting for warehouse access.

**The stronger target I will move toward (Week 5+):**
A **future-window label**: features from the prior 90 days → decline or recovery over the *next* 30 days. This is a true prediction task rather than a snapshot description.

**Why the proxy is safe to start with:**
- It contains no leakage (it is computed from the same window as the features, but the starter pipeline already excludes `trend_direction` and `trend_pct` from features).
- It is honest about its limitation: it finds pages that *look* declining now, not pages guaranteed to decline tomorrow.

In [7]:
import pandas as pd
url = "https://raw.githubusercontent.com/flyrank-bih/flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(url)

# Show the proxy label distribution
label = (df['trend_direction'] == 'down').astype(int)
print(f"Proxy label: declining = {label.sum():,} / {len(df):,} ({label.mean():.1%})")
print("This is the target the starter model predicts. It is a proxy for 'pages showing decline signals now'.")

Proxy label: declining = 16,262 / 30,000 (54.2%)
This is the target the starter model predicts. It is a proxy for 'pages showing decline signals now'.


## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Precision@50**

I defend Precision@50 because the real decision is not "classify every page correctly" — it is "if a reviewer only has time to check 50 pages, how many of those 50 are actually worth checking?"

- The baseline rule achieved Precision@50 ≈ **0.24** (about 12 out of 50 top recommendations were truly declining).
- The random forest achieved Precision@50 ≈ **0.74** (about 37 out of 50).
- That is a **3× improvement** in reviewer productivity — the same human effort yields three times as many real problems found.

I also track **ROC AUC** as a secondary check (it measures overall ranking quality), but Precision@50 is the primary metric because it directly matches the team's capacity. A reviewer cannot act on 30,000 pages; they can act on 50.

In [8]:
# Starter pipeline results from outputs/model_results.json
results = {
    "baseline_rules": {"Precision@50": 0.240, "ROC_AUC": 0.627},
    "logistic_regression": {"Precision@50": 0.400, "ROC_AUC": 0.700},
    "decision_tree": {"Precision@50": 0.540, "ROC_AUC": 0.742},
    "random_forest": {"Precision@50": 0.740, "ROC_AUC": 0.750}
}

print("Precision@50 comparison (higher = more real declines in top 50):")
for model, metrics in results.items():
    print(f"  {model:20s}: {metrics['Precision@50']:.3f}")

print(f"\nRandom forest lift over baseline: {results['random_forest']['Precision@50'] / results['baseline_rules']['Precision@50']:.1f}x")
print("This is why Precision@50 is the metric I optimize for.")

Precision@50 comparison (higher = more real declines in top 50):
  baseline_rules      : 0.240
  logistic_regression : 0.400
  decision_tree       : 0.540
  random_forest       : 0.740

Random forest lift over baseline: 3.1x
This is why Precision@50 is the metric I optimize for.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

**One row = one content item (one pseudonymized page).**

The starter dataset `content_refresh_anonymized.csv` has 30,000 rows, and each row represents a single page from a single client. All metrics are aggregated over a trailing 90-day window. This is the grain I model at — I am not predicting per-client behavior or per-day fluctuations; I am scoring individual pages for review priority.

In [9]:
import pandas as pd

url = "https://raw.githubusercontent.com/flyrank-bih/flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(url)

print(f"Shape: {df.shape} → {df.shape[0]:,} rows, {df.shape[1]} columns")
print(f"Unique content items: {df['content_id'].nunique():,}")
print(f"Unique clients: {df['client_id'].nunique()}")
print("\nOne row = one content item. Confirmed: content_id is unique per row.")
print("\nSample row (key fields only):")
print(df[['content_id', 'client_id', 'impressions_90d', 'clicks_90d',
          'ctr', 'avg_position', 'content_age_days', 'trend_direction']].head(1).T)

Shape: (30000, 44) → 30,000 rows, 44 columns
Unique content items: 30,000
Unique clients: 32

One row = one content item. Confirmed: content_id is unique per row.

Sample row (key fields only):
                                     0
content_id        content_304f48230142
client_id            client_f369cb89fc
impressions_90d                   3803
clicks_90d                          29
ctr                               0.76
avg_position                      10.6
content_age_days                   187
trend_direction                   down


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed rule cannot weigh **interacting signals** dynamically.

The starter baseline uses a transparent but rigid formula:
`0.40 × visibility + 0.30 × freshness + 0.25 × position + 0.05 × depth`

This works for obvious cases (e.g., a very old page with lots of impressions), but it fails when signals conflict:

- A page with **high impressions** but **rising CTR** and **recent update** — the rule might still flag it because of visibility weight, but it is actually healthy.
- A page with **moderate impressions**, **low CTR**, **page-one position**, and **declining trend** — the rule misses the interaction because no single threshold captures "position is good but CTR is bad AND trend is down."
- A page with **thin word count** but **strong engagement** — the rule might call it "thin," but users clearly like it.

A random forest learns these **combinations** from the data. It does not need a human to pre-define which signal matters more — it discovers that, for example, `low CTR + high position + old age` is a stronger decline predictor than any of those signals alone. The 3× Precision@50 lift proves the pattern is too messy for an if-statement.

In [10]:
# Show one example where signals conflict — a case a fixed rule would mishandle
import pandas as pd
url = "https://raw.githubusercontent.com/flyrank-bih/flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(url)

# Find a page that is declining despite NOT being extremely old or extremely visible
# (the kind of case a simple rule misses)
mask = (
    (df['trend_direction'] == 'down') &
    (df['content_age_days'] < 180) &           # not extremely old
    (df['impressions_90d'] >= 500) &           # visible enough to matter
    (df['impressions_90d'] < 3000) &           # but not a giant
    (df['ctr'] < 0.5) &                        # underperforms on CTR
    (df['avg_position'] > 0) &
    (df['avg_position'] <= 10)                 # ranks well position-wise
)

example = df[mask][['content_age_days', 'impressions_90d', 'ctr',
                    'avg_position', 'trend_direction', 'days_since_last_update']].head(1)

print("Example of a 'messy' page — signals conflict:")
print(example.T)
print("\nA fixed rule would need separate thresholds for age, impressions, CTR, position, AND freshness.")
print("A model learns that THIS combination predicts decline — no human had to write that rule.")

Example of a 'messy' page — signals conflict:
                          46
content_age_days         144
impressions_90d         1193
ctr                     0.08
avg_position             6.1
trend_direction         down
days_since_last_update   104

A fixed rule would need separate thresholds for age, impressions, CTR, position, AND freshness.
A model learns that THIS combination predicts decline — no human had to write that rule.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.